# Trace an ML repository to its research papers with MCP

## Overview

Build the retrieval component of a research agent: start with a public GitHub
repository and return its recorded paper relationships with sources to inspect.
The example uses FlashAttention and Lacuna's public Model Context Protocol (MCP)
server. It requires Python 3.11+, basic async Python, and internet access.
No model API key or Lacuna account is needed.

You will learn to discover a resource, verify its identity, follow publication
links, and preserve evidence instead of attributing a paper from search rank.
The workflow uses explicit Python decisions. It does not call a language model.

1. Connect to MCP and decode tool responses.
2. Resolve the repository and fetch its linked papers.
3. Compare with a single paper search.
4. Test a missing match and adapt the workflow to another repository.


## Detailed Explanation

The goal is a source trail, not an unconstrained literature summary. State consists
of the target repository, search candidates, selected resource, publication edges,
and fetched papers. The tools are `search_lacuna`, `get_resource`, and `get_paper`.

The controller requires one exact repository identity among the candidates before
following any paper links. It then fetches at most three papers by default. That
bounds the workflow to at most five tool calls. A missing or ambiguous identity
stops the workflow. No paper edges produces an empty result with an explicit limit
on what that means.

An LLM-based agent can use the same tools and decisions. Keeping the source-tracing
step explicit makes its behavior inspectable without a paid model call.

![Code-to-paper retrieval architecture](../assets/code-to-paper.svg)


## Required Packages

Install the tested MCP client version into this notebook's kernel. If your kernel
has already imported another MCP version, restart it after this cell.


In [ ]:
%pip install -q "mcp==2.3.0"

## Implementation

### Imports and public endpoint

Only public search terms and repository identities are sent. No local code is read
or uploaded. Each MCP session closes when its async context exits.


In [ ]:
import json
from datetime import UTC, datetime
from time import perf_counter
from typing import Any
from urllib.parse import urlparse

from mcp import Client

MCP_URL = "https://lacuna.tiptreesystems.com/api/mcp"

### Resolve identity before attribution

Compare owner/name against returned identity fields. A recorded renamed repository
can have an old provider key and a current title. Multiple distinct matches are an
ambiguity, even when one has a higher score. This check does not independently
verify the repository's authorship; the final evidence links still need inspection.


In [ ]:
def github_key(value: str) -> str:
    """Normalize a public GitHub repository URL or owner/name, without guessing aliases."""
    value = value.strip()
    if "://" in value:
        parsed = urlparse(value)
        if parsed.scheme not in {"http", "https"} or parsed.hostname != "github.com":
            raise ValueError("Use a public github.com repository URL or owner/name.")
        value = parsed.path.strip("/")
    key = value.rstrip("/").removesuffix(".git").lower()
    if len(key.split("/")) != 2 or not all(key.split("/")):
        raise ValueError("Use owner/name, not a file URL or a bare repository name.")
    return key


def select_resource(results: list[dict[str, Any]], repository: str) -> dict[str, Any]:
    """Require one identity match; a high search rank alone is not enough."""
    wanted = github_key(repository)
    matches = {}
    for item in results:
        if item.get("provider") != "github":
            continue
        for field in ("provider_key", "title", "canonical_url", "url"):
            try:
                matched = github_key(item.get(field) or "") == wanted
            except ValueError:
                continue
            if matched and item.get("id"):
                matches[item["id"]] = item
    if len(matches) != 1:
        candidates = [item.get("title", item.get("id")) for item in results]
        raise ValueError(
            f"Expected one exact repository match, found {len(matches)}. "
            f"Inspect these candidates or refine --query: {candidates}"
        )
    return next(iter(matches.values()))

### Decode tool results and preserve errors

MCP may supply structured content or JSON text. A service error must remain an
error; turning it into an empty list would falsely suggest no papers were found.


In [ ]:
async def call_json(client: Client, name: str, arguments: dict[str, Any]) -> dict[str, Any]:
    """Read structured MCP data or its JSON text fallback, surfacing tool errors."""
    result = await client.call_tool(name, arguments)
    if result.is_error:
        message = "\n".join(block.text for block in result.content if block.type == "text")
        raise RuntimeError(f"{name}: {message}")
    data = result.structured_content
    if data is None:
        text = "\n".join(block.text for block in result.content if block.type == "text")
        data = json.loads(text)
    if not isinstance(data, dict):
        raise ValueError(f"{name} returned a non-object payload")
    return data

### Follow the resource-to-paper relationships

Search first, resolve one repository, and follow returned IDs. The result retains
the relationship label and source links. It also records when Lacuna collected the
resource so a reader can judge freshness. This function never invents a paper ID.


In [ ]:
async def trace_repository(
    repository: str = "Dao-AILab/flash-attention",
    query: str = "FlashAttention",
    max_papers: int = 3,
    *,
    server: Any = MCP_URL,
) -> dict[str, Any]:
    """Search, resolve a repository, and follow a bounded set of recorded paper edges."""
    github_key(repository)
    if not 1 <= max_papers <= 10:
        raise ValueError("max_papers must be between 1 and 10")
    started = perf_counter()
    calls = []
    async with Client(server, read_timeout_seconds=60) as client:

        async def read(name: str, **arguments: Any) -> dict[str, Any]:
            calls.append({"tool": name, "arguments": arguments})
            return await call_json(client, name, arguments)

        search = await read(
            "search_lacuna", query=query, search_type="resource", resource_kind="codebase", limit=5
        )
        selected = select_resource(search.get("results", []), repository)
        resource = await read("get_resource", resource_id_or_url=selected["id"])
        edges = resource.get("publications") or []
        papers = []
        seen = set()
        for edge in edges:
            paper_id = edge.get("paper_id")
            if not paper_id or paper_id in seen:
                continue
            seen.add(paper_id)
            paper = await read(
                "get_paper", artifact_id_or_url=paper_id, view="context", figure_limit=0
            )
            papers.append(
                {
                    "paper_id": paper_id,
                    "title": paper.get("title"),
                    "year": paper.get("year"),
                    "lacuna_url": paper.get("url"),
                    "relationship": edge.get("relationship"),
                    "evidence_url": edge.get("evidence_url"),
                    "identifiers": paper.get("identifiers", {}),
                }
            )
            if len(papers) == max_papers:
                break

    return {
        "retrieved_at": datetime.now(UTC).isoformat(),
        "repository": repository,
        "resource_id": selected["id"],
        "resource_url": selected.get("context_url"),
        "external_url": resource.get("url"),
        "resource_collected_at": resource.get("collected_at"),
        "papers": papers,
        "publication_edges_returned": len(edges),
        "papers_fetched": len(papers),
        "tool_calls": calls,
        "elapsed_seconds": round(perf_counter() - started, 3),
        "note": (
            "Relationships are records in Lacuna. Inspect the linked sources to verify them. "
            "An empty or bounded result does not establish that no other papers exist."
        ),
    }

## Usage Example

Run the FlashAttention example. Jupyter supports top-level `await`; in a Python
script use `asyncio.run(...)`. Live results can change as the corpus is updated.


In [ ]:
trace = await trace_repository()
print(json.dumps(trace, indent=2))

### Interpret the evidence

The recorded example includes `official_code_for` for *FlashAttention: Fast and
Memory-Efficient Exact Attention with IO-Awareness*. Open the returned repository
and original paper links to confirm the relationship yourself. A Lacuna summary
is derived material, not a quotation from the original paper.

The result can include fewer papers than a resource search count suggests. Do not
interpret either count as an exhaustive bibliography.


In [ ]:
for paper in trace["papers"]:
    print(f"{paper['relationship']}: {paper['title']}")
    print("Lacuna:", paper["lacuna_url"])
    print("Relationship evidence:", paper["evidence_url"])
    print("Original paper:", paper["identifiers"].get("pdf_url", "Not provided"))

## Comparison

A single paper search returns relevant candidates. It does not, by itself, establish
which repository implements them. Compare the actual tool-call counts and elapsed
times below. This is one live example, not an accuracy or latency benchmark.
Requests run sequentially, and server caching or load can affect the timings.


In [ ]:
started = perf_counter()
async with Client(MCP_URL, read_timeout_seconds=60) as client:
    baseline = await call_json(
        client,
        "search_lacuna",
        {
            "query": "FlashAttention",
            "search_type": "paper",
            "limit": 5,
        },
    )
baseline_seconds = round(perf_counter() - started, 3)
comparison = [
    {
        "method": "one paper search",
        "tool_calls": 1,
        "seconds": baseline_seconds,
        "items_returned": len(baseline.get("results", [])),
        "evidence": "paper candidates; repository relationship not checked",
    },
    {
        "method": "resource-to-paper traversal",
        "tool_calls": len(trace["tool_calls"]),
        "seconds": trace["elapsed_seconds"],
        "items_returned": len(trace["papers"]),
        "evidence": "recorded relationships plus sources to verify",
    },
]
print(json.dumps(comparison, indent=2))
print("Baseline candidates:")
for item in baseline.get("results", []):
    print("-", item.get("title"))

### Check the missing-match behavior

This offline case makes sure the controller does not take an unrelated first result.
The expected outcome is a clear refusal to attribute papers to the target repository.


In [ ]:
unrelated = [{"id": "example-only", "provider": "github", "title": "someone/other-code"}]
try:
    select_resource(unrelated, "Dao-AILab/flash-attention")
except ValueError as exc:
    print("Stopped as intended:", exc)
else:
    raise RuntimeError("The workflow attributed an unrelated repository.")

### Exercise: trace a second repository

Choose a public ML implementation you know. Set `exercise_repository` to its
owner/name and `exercise_query` to its project name. Inspect the paper relationship
and original sources. If no match exists, report that limitation rather than
loosening the identity check. The cell is idle until you supply both inputs.


In [ ]:
exercise_repository = None
exercise_query = None
if exercise_repository and exercise_query:
    exercise = await trace_repository(exercise_repository, exercise_query, max_papers=2)
    print(json.dumps(exercise, indent=2))
else:
    print("Set the two public repository inputs above to run the exercise.")

## Additional Considerations

- **Coverage:** Lacuna focuses on ML/AI. An absent resource or edge is missing
  evidence, not proof that there are no associated papers.
- **Freshness:** resource metadata can lag the upstream repository. Follow the
  evidence links and inspect the recorded collection timestamp.
- **Versions:** different paper IDs can represent versions of one work. Compare
  DOI/arXiv identity and version metadata before counting distinct contributions.
- **Meaning:** `official_code_for` is a recorded relationship. Other labels may
  mean a reimplementation, dataset, or mention; preserve the actual label.
- **Service failures:** tool errors and timeouts are surfaced. Retry later for
  transient failures; do not treat them as negative research results.
- **Agent extension:** use the source trail as evidence for a coding agent's next
  step. Keep retrieved README text and paper content separate from instructions.
  Ask the agent to distinguish verified sources from its own interpretation.

## References

- [Lacuna Research MCP: server, tools, and research skills](https://github.com/tiptreesystems/lacuna-research-mcp)
- [MCP Python SDK](https://github.com/modelcontextprotocol/python-sdk)
- [FlashAttention repository](https://github.com/Dao-AILab/flash-attention)
- [FlashAttention paper](https://arxiv.org/abs/2205.14135)
- [Lacuna: A Research Map for Machine Learning](https://arxiv.org/abs/2606.26246)

The implementation in this notebook is original to Lacuna Research MCP and is
provided under that repository's MIT license.
